# Training LifeMate Assistant with Reinforcement Learning: LFM3-1.7B + GRPO with TRL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com) [![GitHub](https://img.shields.io/badge/GitHub-ZL26--TASK1-181717?logo=github)](https://github.com/EsperCS/ZL26-TASK1) [![Zalo AI Challenge](https://img.shields.io/badge/ZaloAI--2026-Task1--LifeMate-blue)](https://zaic.zalo.ai/)

Notebook này giữ nguyên 100% cấu trúc, thiết lập thư viện và logic so khớp của [Cookbook Chính Thức của Liquid AI](https://github.com/Liquid4All/cookbook/blob/main/finetuning/notebooks/lfm3_grpo_with_trl.ipynb), chỉ lắp tập dữ liệu từ **Zalo AI Challenge 2026 (Task 1: LifeMate)** lấy trực tiếp từ Google Drive.

## 🎯 What you'll find

- **The app**: Danh mục 148 công cụ LifeMate (`train_tools_catalog.json`) nạp vào mọi prompt.
- **A verifiable reward**: Hàm thưởng kiểm chứng so khớp từng tham số với gold calls theo đúng chuẩn của Liquid AI.
- **GRPO + LoRA**: Mô hình LFM3-1.7B tự học qua thuật toán Group Relative Policy Optimization (num_generations=8, generation_batch_size=8, r=32) chống OOM tuyệt đối trên Colab A100.
- **Evaluation**: Đánh giá độ chính xác so khớp hoàn toàn (exact match) trước và sau khi huấn luyện.

# 📦 Installation & Setup

Kết nối Google Drive để nạp dữ liệu cuộc thi (`train+public_test.zip`) và cố định đúng các phiên bản thư viện đã được Liquid AI kiểm thử.

In [ ]:
# 1. Kết nối Google Drive để lấy dữ liệu
from google.colab import drive
drive.mount('/content/drive')

# 2. Gỡ bỏ torchvision và cudf xung đột trên Colab để tránh lỗi C++ operator
!pip uninstall -y torchvision cudf-cu13 cudf-polars-cu13 -q

# 3. Cài đặt các thư viện y xì đúc như notebook của Liquid AI
!pip install -qqq "torch==2.14.1" "torchao==0.18.0" "trl==1.16.0" "transformers==5.20.2" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.1.0" "huggingface-hub==2.3.0" "tokenizers==0.23.3" "safetensors==0.8.0" --progress-bar off


## 📂 Giải Nén Dữ Liệu Từ Google Drive

Giải nén file `train+public_test.zip` từ đường dẫn `Drive của tôi > ZaloAI_2026 > task1_lifemate` vào thư mục `/content/data`.

In [ ]:
import os
from pathlib import Path

# Đường dẫn file zip chính xác từ Google Drive của bạn
zip_path = Path("/content/drive/MyDrive/ZaloAI_2026/task1_lifemate/train+public_test.zip")

# Fallback tự tìm nếu nằm ở thư mục khác trên Drive
if not zip_path.exists():
    candidates = [
        Path("/content/drive/MyDrive/ZaloAI_2026_ASR/train+public_test.zip"),
        Path("/content/drive/MyDrive/task1_lifemate/train+public_test.zip"),
        Path("/content/drive/MyDrive/train+public_test.zip"),
    ]
    for c in candidates:
        if c.exists():
            zip_path = c
            break

if not zip_path.exists():
    raise FileNotFoundError(f"Không tìm thấy file {zip_path} trên Google Drive! Bạn hãy kiểm tra lại kết nối Drive.")

print(f"📦 Tìm thấy file zip: {zip_path}")
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Giải nén cực nhanh bằng lệnh shell Linux
!unzip -q -o "{zip_path}" -d /content/data

print("✅ Đã giải nén thành công vào /content/data! Danh sách tệp tin:")
!ls -lh /content/data


In [ ]:
import json
import os
import torch
import transformers
import trl
import peft

os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")
print(f"🎛️ PEFT version: {peft.__version__}")

# LFM3-1.7B is released in bfloat16. GPUs without bfloat16 support load it in float16 instead:
# the conversion is exact for almost every weight, and the LoRA adapters train in float32 either way.
use_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"🔢 Precision: {dtype}")

# Loading the model from Transformers 🤗

LFM3-1.7B ships its modeling code with the checkpoint, so we load it with `trust_remote_code=True`. We pad on the left, so that batched generation continues each prompt right where it ends.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "LiquidAI/LFM3-1.7B"

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, padding_side="left")

print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, device_map="auto", trust_remote_code=True)
print("✅ Model loaded successfully!")

# 📱 The app: 148 tools (Zalo LifeMate Ecosystem)

Thay vì tải bộ tool của Liquid AI, chúng ta nạp danh mục 148 công cụ LifeMate từ thư mục `/content/data` vừa giải nén và tự động suy luận kiểu dữ liệu (`kinds`) để dùng cho hàm thưởng so khớp.

In [ ]:
from collections import Counter
from pathlib import Path
import json

# Tự động quét tìm file train_tools_catalog.json trong thư mục vừa giải nén
tool_file = next(Path("/content/data").rglob("train_tools_catalog.json"), None)
if not tool_file:
    tool_file = next(Path("/content").rglob("train_tools_catalog.json"), None)

if not tool_file:
    raise FileNotFoundError("Không tìm thấy file train_tools_catalog.json trong /content/data! Hãy chạy cell giải nén ở trên.")

print(f"📄 Đang nạp danh mục công cụ từ: {tool_file}")
raw_catalog = json.loads(tool_file.read_text(encoding="utf-8"))

tools = []
kinds = {}

# Luôn có công cụ hệ thống sys.ask_missing_info
tools.append({
    "type": "function",
    "function": {
        "name": "sys.ask_missing_info",
        "description": "Hỏi lại người dùng khi thiếu thông tin bắt buộc để thực hiện tác vụ.",
        "parameters": {
            "type": "object",
            "properties": {
                "tool_name": {"type": "string", "description": "Tên tool dự định gọi"},
                "params": {"type": "array", "items": {"type": "string"}, "description": "Danh sách tham số còn thiếu"}
            },
            "required": ["tool_name", "params"]
        }
    }
})
kinds["sys.ask_missing_info"] = {"tool_name": "name", "params": "list_name"}

# Nạp toàn bộ 147 công cụ từ 46 ứng dụng trong catalog LifeMate
for app in raw_catalog.get("apps", []):
    for t in app.get("tools", []):
        t_name = t["name"]
        params = t.get("parameters", {"type": "object", "properties": {}})
        tools.append({
            "type": "function",
            "function": {
                "name": t_name,
                "description": t.get("description", ""),
                "parameters": params
            }
        })
        
        # Tự động suy luận kinds phục vụ so khớp trong cùng cấu trúc Liquid AI
        t_kinds = {}
        props = params.get("properties", {})
        for p_name, p_spec in props.items():
            p_type = p_spec.get("type", "string")
            p_fmt = p_spec.get("format", "")
            if p_fmt == "date-time" or "datetime" in p_name:
                kind = "datetime"
            elif p_fmt == "date" or "date" in p_name:
                kind = "date"
            elif p_fmt == "time" or ("time" in p_name and "times" not in p_name):
                kind = "time"
            elif p_type == "boolean":
                kind = "bool"
            elif p_type in ("integer", "number"):
                kind = "int" if p_type == "integer" else "number"
            elif "enum" in p_spec:
                kind = "enum"
            elif p_type == "array":
                item_spec = p_spec.get("items", {})
                if "enum" in item_spec:
                    kind = "list_enum"
                elif item_spec.get("type") == "string":
                    kind = "list_name" if ("id" in p_name or "name" in p_name) else "list_text"
                else:
                    kind = "list_text"
            elif "id" in p_name or "phone" in p_name or "code" in p_name:
                kind = "code"
            elif "name" in p_name or "title" in p_name:
                kind = "name"
            else:
                kind = "text"
            t_kinds[p_name] = kind
        kinds[t_name] = t_kinds

print(f"🧰 {len(tools)} tools loaded, for example:")
for tool in tools[:3]:
    print(json.dumps(tool["function"], ensure_ascii=False)[:300])


## The requests (Zalo LifeMate Dataset)

Chúng ta nạp `train.jsonl` từ `/content/data` vừa giải nén, ánh xạ chuẩn vào cấu trúc `request`, `context`, `calls`, `category` đúng y xì định dạng của Liquid AI:

- `train`: 1,390 câu lệnh thực tế từ người dùng.
- `test`: 155 câu lệnh dùng để đánh giá độ chính xác.
- `showcase`: 5 ví dụ chọn lọc trải rộng các loại yêu cầu (`single`, `multi`, `clarify`, `no_tool`).

In [ ]:
from datasets import Dataset, DatasetDict
import random

# Tự động quét tìm file train.jsonl trong thư mục vừa giải nén
data_file = next(Path("/content/data").rglob("train.jsonl"), None)
if not data_file:
    data_file = next(Path("/content").rglob("train.jsonl"), None)

if not data_file:
    raise FileNotFoundError("Không tìm thấy file train.jsonl trong /content/data! Hãy chạy cell giải nén ở trên.")

print(f"📄 Đang nạp dữ liệu từ: {data_file}")
raw_records = []
with open(data_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            raw_records.append(json.loads(line))

processed_records = []
for r in raw_records:
    calls = r.get("label", {}).get("tool_calls", [])
    if not calls:
        category = "no_tool"
    elif any(c.get("name") == "sys.ask_missing_info" for c in calls):
        category = "clarify"
    elif len(calls) > 1:
        category = "multi"
    else:
        category = "single"
        
    ctx = r.get("context", {})
    current_time_str = ctx.get("current_time", "2026-11-28T07:10:00+07:00")
    dt_clean = current_time_str.split("+")[0] if "+" in current_time_str else current_time_str
    
    clean_ctx = {
        "datetime": dt_clean,
        "timezone": ctx.get("timezone", "Asia/Ho_Chi_Minh"),
        "locale": "vi-VN",
        "user_profile": json.dumps(ctx.get("user_profile", {}), ensure_ascii=False) if ctx.get("user_profile") else "",
    }
    
    processed_records.append({
        "request": r.get("audio_text", ""),
        "context": clean_ctx,
        "calls": json.dumps(calls, ensure_ascii=False),
        "category": category,
        "language": "vi",
    })

# Chia train (90%) và test (10%)
random.seed(42)
random.shuffle(processed_records)
n_train = int(len(processed_records) * 0.9)

dataset = DatasetDict({
    "train": Dataset.from_list(processed_records[:n_train]),
    "test": Dataset.from_list(processed_records[n_train:]),
    "showcase": Dataset.from_list(processed_records[n_train:n_train + 5]),
})

print(dataset)
print(Counter(dataset["train"]["category"]))

for category in ["multi", "clarify"]:
    example = next(e for e in dataset["train"] if e["category"] == category)
    print(f"\n🗂️ {category} ({example['language']}, {example['context']['datetime']}): {example['request']}")
    print(f"🎯 Gold calls: {example['calls']}")


## Build the prompts

The system prompt holds the rules and the context, and the chat template adds the 148 tool schemas. LFM3 writes tool calls in XML, and the tokenizer knows how to parse them back into structured calls with `tokenizer.parse_response`.

We use LFM3's non-reasoning mode (`enable_thinking=False`). Since every prompt carries the same catalog, we render the prompts ourselves and give TRL plain text prompts.

In [ ]:
from datetime import datetime

SYSTEM_PROMPT = """You are a helpful assistant running on the user's phone. You can call the phone's tools.
Current date and time: {weekday}, {date} {time} ({timezone}). User locale: {locale}.
{user_profile_info}
Rules:
- Call every tool needed to complete the request, with arguments taken from the request and the context above. Fill optional arguments only when the user gives them; never invent values.
- Resolve relative dates and times ("tomorrow", "on Friday", "in 20 minutes") against the current date and time. Times are local, in the formats the tool descriptions give.
- If the request needs a tool but a required detail is missing, ask a short clarifying question using sys.ask_missing_info instead of calling other tools.
- Answer small talk and general-knowledge questions directly, without tools.
- If no tool can do what is asked, say so briefly instead of calling a different tool. If only part of the request is possible, do that part and say what you could not do.
- Keep free-text arguments (messages, notes, search queries) in Vietnamese as requested."""

def build_prompt(example):
    now = datetime.fromisoformat(example["context"]["datetime"])
    up = example["context"].get("user_profile", "")
    user_profile_info = f"User profile: {up}" if up else ""
    system = SYSTEM_PROMPT.format(
        weekday=now.strftime("%A"),
        date=now.strftime("%Y-%m-%d"),
        time=now.strftime("%H:%M"),
        user_profile_info=user_profile_info,
        **example["context"]
    )
    messages = [{"role": "system", "content": system}, {"role": "user", "content": example["request"]}]
    return tokenizer.apply_chat_template(messages, tools=tools, add_generation_prompt=True, tokenize=False, enable_thinking=False)

prompt = build_prompt(dataset["train"][0])
print(f"📏 Prompt length: {len(tokenizer(prompt, add_special_tokens=False).input_ids):,} tokens")
print(prompt[-1200:])


## Compare calls with the gold calls

A prediction is correct when it makes exactly the gold calls, in any order. Arguments are compared by kind:

- numbers, booleans, and enums exactly
- dates and times as values (`19:00` is `19:00:00`)
- codes such as currencies and phone numbers, without spaces or dashes
- names, without case or accents
- free text (a message body, a search query), with some tolerance for wording

Optional arguments left at their default value count as absent.

In [ ]:
import ast
import itertools
import math
import operator
import re
import unicodedata
from datetime import date, time
from difflib import SequenceMatcher

def norm(s):
    s = unicodedata.normalize("NFKC", str(s)).casefold()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = "".join(" " if unicodedata.category(c)[0] in "PSZ" else c for c in unicodedata.normalize("NFC", s))
    return " ".join(s.split())

def contained(a, b):
    short, long_ = sorted((a, b), key=len)
    return bool(short) and short in long_ and len(short) / len(long_) >= 0.5

def name_equal(a, b):
    a, b = norm(a), norm(b)
    return a == b or contained(a, b)

def text_similar(a, b):
    a, b = norm(a), norm(b)
    return a == b or contained(a, b) or (bool(a) and bool(b) and SequenceMatcher(None, a, b).ratio() >= 0.6)

def parse_when(value, kind):
    value = str(value).strip()
    patterns = {"datetime": r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}(:\d{2})?", "date": r"\d{4}-\d{2}-\d{2}", "time": r"\d{1,2}:\d{2}(:\d{2})?"}
    if not re.fullmatch(patterns[kind], value):
        return None
    return {"datetime": datetime, "date": date, "time": time}[kind].fromisoformat(value.zfill(5) if kind == "time" and len(value) == 4 else value)

OPERATORS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
             ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calculate(expression):
    """Safely evaluate an arithmetic expression, or return None."""
    def walk(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return OPERATORS[type(node.op)](walk(node.left), walk(node.right))
        if isinstance(node, ast.UnaryOp):
            return OPERATORS[type(node.op)](walk(node.operand))
        raise ValueError
    try:
        return float(walk(ast.parse(str(expression).replace("^", "**").replace("×", "*").replace("÷", "/"), mode="eval").body))
    except Exception:
        return None

def same_list(gold, pred, equal):
    return len(gold) == len(pred) and any(all(equal(g, p) for g, p in zip(gold, perm)) for perm in itertools.permutations(pred))

def same_value(kind, gold, pred):
    try:
        if kind == "bool":
            return isinstance(pred, bool) and gold == pred
        if kind in ("int", "number"):
            return isinstance(pred, (int, float)) and not isinstance(pred, bool) and abs(float(gold) - float(pred)) < 1e-6
        if kind == "enum":
            return isinstance(pred, str) and gold == pred.strip().lower()
        if kind == "code":
            return isinstance(pred, str) and re.sub(r"[\s\-().]", "", str(gold).casefold()) == re.sub(r"[\s\-().]", "", pred.casefold())
        if kind in ("datetime", "date", "time"):
            return parse_when(pred, kind) is not None and parse_when(gold, kind) == parse_when(pred, kind)
        if kind == "expr":
            g, p = calculate(gold), calculate(pred)
            return g is not None and p is not None and math.isclose(g, p, rel_tol=1e-9, abs_tol=1e-9)
        if kind == "name":
            return isinstance(pred, str) and name_equal(gold, pred)
        if kind == "text":
            return isinstance(pred, str) and text_similar(gold, pred)
        if kind == "list_enum":
            return isinstance(pred, list) and sorted(gold) == sorted(str(x).lower() for x in pred)
        if kind == "list_name":
            return isinstance(pred, list) and same_list(gold, [str(x) for x in pred], name_equal)
        if kind == "list_text":
            return isinstance(pred, list) and same_list(gold, [str(x) for x in pred], text_similar)
    except (TypeError, ValueError, OverflowError, ZeroDivisionError):
        return False
    return False

schemas = {tool["function"]["name"]: tool["function"]["parameters"] for tool in tools}

def canonical(name, arguments):
    """Fill schema defaults, drop empty optional values, and treat end_date == date as a single day."""
    schema = schemas[name]
    required = set(schema.get("required", []))
    out = {k: v for k, v in (arguments or {}).items() if k in required or v not in (None, "", [])}
    for k, prop in schema["properties"].items():
        if k not in out and "default" in prop:
            out[k] = prop["default"]
    if "end_date" in out and out.get("end_date") == out.get("date"):
        del out["end_date"]
    return out

def same_call(gold, pred):
    if gold["name"] != pred.get("name"):
        return False
    g, p = canonical(gold["name"], gold.get("arguments")), canonical(gold["name"], pred.get("arguments"))
    return set(g) == set(p) and all(k in kinds[gold["name"]] and same_value(kinds[gold["name"]][k], g[k], p[k]) for k in g)

def calls_match(gold, pred):
    """True when the prediction makes exactly the gold calls, in any order."""
    if len(gold) != len(pred) or any(call.get("name") not in schemas for call in pred):
        return False
    return any(all(same_call(g, p) for g, p in zip(gold, perm)) for perm in itertools.permutations(pred))

def predicted_calls(completion_ids, prompt):
    """Parse a completion into a list of {"name", "arguments"} calls."""
    message = tokenizer.parse_response(completion_ids, prefix=prompt, tools=tools)
    return [call["function"] for call in message.get("tool_calls") or []]

# A quick check on a gold example
example = dataset["train"][0]
gold = json.loads(example["calls"])
print(calls_match(gold, gold), calls_match(gold, []))

# 📏 Evaluate the model before training

We generate one answer per request and count exact matches, per category. To keep the notebook fast, we evaluate on 200 requests of each test split.

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate(model, split, n=150, batch_size=8):
    model.eval()
    examples = dataset[split].shuffle(seed=42).select(range(min(n, len(dataset[split]))))
    actual_n = len(examples)
    correct = {}
    for start in tqdm(range(0, actual_n, batch_size), desc=f"Evaluating {split}"):
        batch = examples.select(range(start, min(start + batch_size, actual_n)))
        prompts = [build_prompt(e) for e in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        for output, prompt, example in zip(outputs[:, inputs["input_ids"].shape[1]:], prompts, batch):
            output = output[output != tokenizer.pad_token_id]
            correct.setdefault(example["category"], []).append(calls_match(json.loads(example["calls"]), predicted_calls(output, prompt)))
    results = {category: sum(c) / len(c) for category, c in correct.items()}
    results["all"] = sum(sum(c) for c in correct.values()) / actual_n if actual_n > 0 else 0.0
    return results

def show(results, title):
    print(title)
    for split, r in results.items():
        print(f"   {split:<12} exact match {r['all']:.1%} | " + " | ".join(f"{k} {v:.0%}" for k, v in sorted(r.items()) if k != 'all'))

baseline = {split: evaluate(model, split) for split in ["test"]}
show(baseline, "📏 Before training")


# 🎯 Group Relative Policy Optimization (GRPO + LoRA)

## The reward

For each request, GRPO samples a group of answers and compares their rewards: answers that score above the group's average are reinforced, the others discouraged. Our reward is:

- **1.0** when the calls exactly match the gold calls (for requests where the right answer is to make no call, when the model makes no call)
- **0.2 × F1** of the tool names otherwise, a small partial credit for picking the right tools with wrong arguments, which helps early in training

TRL passes the dataset columns (here `calls`) and the completion token ids to the reward function.

In [ ]:
def tool_name_f1(gold, pred):
    gold_names, pred_names = Counter(c["name"] for c in gold), Counter(c["name"] for c in pred)
    overlap = sum((gold_names & pred_names).values())
    return 0.0 if overlap == 0 else 2 * overlap / (sum(gold_names.values()) + sum(pred_names.values()))

def tool_call_reward(prompts, completion_ids, calls, **kwargs):
    rewards = []
    for prompt, ids, gold in zip(prompts, completion_ids, calls):
        gold, pred = json.loads(gold), predicted_calls(ids, prompt)
        rewards.append(1.0 if calls_match(gold, pred) else 0.2 * tool_name_f1(gold, pred))
    return rewards

## Balance the training requests

Only 15% of the training requests need no tool call (`no_tool` and `clarify`). Since most rewards come from making the right calls, RL alone teaches the model to always call a tool, even when a detail is missing and it should ask. We counter this by showing the no-call requests twice.

In [ ]:
from datasets import concatenate_datasets

no_call = dataset["train"].filter(lambda e: e["category"] in ("no_tool", "clarify"))
train_dataset = concatenate_datasets([dataset["train"], no_call]).shuffle(seed=42)
train_dataset = train_dataset.map(lambda e: {"prompt": build_prompt(e)}, remove_columns=["request", "context"])
print(f"📚 Training requests: {len(train_dataset)}")
print(Counter(train_dataset["category"]))

## Wrap the model with PEFT

We adapt the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), the gated short convolutions (`in_proj`, `out_proj`), and the MLP (`w1`, `w2`, `w3`). LFM3's attention layers also have a per-head output gate, `gate_proj`; adding it to the targets is a reasonable choice.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj", "w1", "w2", "w3",
        # "gate_proj",  # optional: LFM3's per-head attention gate
    ],
    task_type="CAUSAL_LM",
)

## Configure training

- `num_generations=8`: the size of each group. With 32 answers per step, each step trains on 4 requests.
- `generation_batch_size=8`: the answers of one request are generated together. Each prompt is about 8,000 tokens, so generating all 32 answers at once would need much more memory.
- `max_completion_length=256`: tool calls and short answers fit easily.
- `temperature=1.0`: enough variety within a group for the reward to tell answers apart.
- `beta=0.04`: a small KL penalty that keeps the model close to the original one. Without it, the behaviors the reward rarely asks for (like asking a clarifying question) fade as training goes on. With LoRA, TRL gets the reference model by disabling the adapter, so it doesn't load a second copy.

TRL generates with Transformers here. It can also generate with [vLLM](https://huggingface.co/docs/trl/vllm_integration) (`use_vllm=True`), which is much faster when your vLLM version supports the model.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="./LFM3-1.7B-lifemate-assistant",
    max_steps=150,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,  # <- 32 answers per step
    num_generations=8,
    generation_batch_size=8,  # <- generate one group at a time to prevent OOM
    max_completion_length=256,
    temperature=1.0,
    learning_rate=1e-5,
    beta=0.04,  # <- KL penalty towards the original model
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=5,
    logging_steps=5,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=tool_call_reward,
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()


## Launch training

In [ ]:
print("🚀 Starting GRPO training...")
trainer.train()
print("🎉 GRPO training completed!")
print(f"🧮 Peak GPU memory: {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

Let's look at how the reward changed during training (1.0 means every answer in the batch was exactly right):

In [ ]:
for log in trainer.state.log_history:
    if "reward" in log:
        print(f"Step {log['step']:>3}: reward {log['reward']:.2f}, mean length {log['completions/mean_length']:.0f} tokens")

# 📊 Evaluate the trained model

In [ ]:
final = {split: evaluate(trainer.model, split) for split in ["test"]}
show(baseline, "📏 Before training")
show(final, "📏 After training")


# 🔍 Original vs fine-tuned model

To see what changed, we give the same requests to the original model and to the trained model. With LoRA, `disable_adapter()` temporarily turns the adapter off, so we don't need to load the original model again. The `showcase` split holds hand-checked requests for tools that never appeared in training.

In [ ]:
def answer(model, example):
    prompt = build_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    output = model.generate(**inputs, max_new_tokens=512, do_sample=False)[0, inputs["input_ids"].shape[1]:]
    message = tokenizer.parse_response(output, prefix=prompt, tools=tools)
    calls = [call["function"] for call in message.get("tool_calls") or []]
    return calls, (message.get("content") or "").strip()

def describe(calls, text):
    return json.dumps(calls, ensure_ascii=False) if calls else f"(no call) {text[:150]}"

trainer.model.eval()
for example in dataset["showcase"]:
    gold = json.loads(example["calls"])
    with trainer.model.disable_adapter():
        original = answer(trainer.model, example)
    trained = answer(trainer.model, example)
    print(f"\n💬 {example['request']} ({example['language']}, {example['category']})")
    print(f"   🎯 Gold:       {describe(gold, 'ask for the missing detail or answer directly')}")
    print(f"   {'✅' if calls_match(gold, original[0]) else '❌'} Original:   {describe(*original)}")
    print(f"   {'✅' if calls_match(gold, trained[0]) else '❌'} Fine-tuned: {describe(*trained)}")

And the summary over all the evaluation requests:

In [ ]:
print(f"{'exact match':<36}{'original':>10}{'fine-tuned':>12}")
for split, name in [("test", "LifeMate test tools")]:
    print(f"{name:<36}{baseline[split]['all']:>10.1%}{final[split]['all']:>12.1%}")
    for category in sorted(k for k in baseline[split] if k != "all"):
        print(f"{'  ' + category:<36}{baseline[split][category]:>10.0%}{final[split][category]:>12.0%}")


# 💾 Save and merge the model

We save the LoRA adapter, then merge it into the base weights. `save_pretrained` also copies LFM3's modeling code next to the weights, so the merged model loads with `trust_remote_code=True` like the original one.

In [ ]:
# Lưu LoRA adapter và bản model đã merge vào Google Drive để sử dụng vĩnh viễn
save_dir = Path("/content/drive/MyDrive/ZaloAI_2026_ASR/LFM3-1.7B-lifemate-merged")
save_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model("./LFM3-1.7B-lifemate-lora")

merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"💾 Mô hình hoàn chỉnh đã lưu thành công vào Drive: {save_dir}")


# 🚀 Next steps

- **Your own app**: export your tools as function schemas, collect real requests, and label the right calls. The reward only needs a way to compare calls, so you don't need full demonstrations.
- **Train longer**: the accuracy keeps improving after this run (200 steps reach about 80% on the training tools and 76% on the held-out ones). Increase `max_steps`.
- **Multi-turn agents**: `GRPOTrainer` can also run your tools during generation (`tools=[...]` or `environment_factory=...`) and reward the final outcome. See the [TRL documentation](https://huggingface.co/docs/trl/grpo_trainer).